# Validate ResStock loads against EIA-861

This **state-agnostic** template notebook loads ResStock building metadata, utility
assignments, and annual load curves for a given state and release, then (in later
sections) compares weighted residential electricity totals to EIA-861 utility sales.

> **Demonstration state**: CT / `res_2024_amy2018_2`.  
> **To run for another state**, change `STATE` and (if needed) `RESSTOCK_RELEASE`
> in the Parameters cell and restart the kernel.

---

## How to use

1. Set `STATE` (lowercase 2-letter abbreviation) and `RESSTOCK_RELEASE` in Parameters.
2. Optionally set `UPGRADE` (default `"00"`) and `EIA_YEAR` (default `2018`, to match
   ResStock AMY 2018).
3. Restart the kernel and run all cells.

---

## Data sources

| Input | S3 location |
|-------|-------------|
| Metadata (`metadata-sb`) | `s3://data.sb/nrel/resstock/<release>_sb/metadata/state=<ST>/upgrade=<uu>/metadata-sb.parquet` |
| Utility assignment | `s3://data.sb/nrel/resstock/<release>_sb/metadata_utility/state=<ST>/utility_assignment.parquet` |
| Annual load curves | `s3://data.sb/nrel/resstock/<release>/load_curve_annual/state=<ST>/upgrade=<uu>/<ST>_upgrade<uu>_metadata_and_annual_results.parquet` |
| EIA-861 utility stats | `s3://data.sb/eia/861/electric_utility_stats/year=<year>/state=<ST>/data.parquet` |

`load_curve_annual` lives only on the **raw** ResStock release (it is excluded from
`_sb`). Metadata and utility assignment live on the **`_sb`** companion release.
This notebook derives both paths from a single `RESSTOCK_RELEASE` parameter.

---

## Notebook sections

| Section | What it covers |
|---------|---------------|
| **1 — Load data** | Read metadata, utility assignment, and annual load curves from S3; keep the `bldg_id` intersection |
| **2 — Sum by utility** | Weighted electricity totals by `sb.electric_utility` |
| **3 — Compare to EIA** | *(todo)* Join EIA-861 residential sales; ratios and % diffs |
| **4 — Assumptions** | *(todo)* Document limitations and interpretation guidance |

## Parameters

Change `STATE` and `RESSTOCK_RELEASE` here. Everything else derives from these values.

In [1]:
from __future__ import annotations

from typing import cast

import polars as pl
from IPython.display import display

In [2]:
# ── Change these to validate a different state or release ─────────────────────
STATE = "ct"  # lowercase state abbreviation, e.g. "ri", "ny", "ma"
RESSTOCK_RELEASE = "res_2024_amy2018_2"  # base (non-_sb) release name
UPGRADE = "00"  # zero-padded ResStock upgrade id
EIA_YEAR = 2018  # EIA-861 report year (2018 aligns with ResStock AMY 2018)
# ──────────────────────────────────────────────────────────────────────────────

STATE_UPPER = STATE.upper()

# load_curve_annual is on the raw release; metadata + utility_assignment are on _sb
if RESSTOCK_RELEASE.endswith("_sb"):
    RESSTOCK_RELEASE_RAW = RESSTOCK_RELEASE.removesuffix("_sb")
    RESSTOCK_RELEASE_SB = RESSTOCK_RELEASE
else:
    RESSTOCK_RELEASE_RAW = RESSTOCK_RELEASE
    RESSTOCK_RELEASE_SB = f"{RESSTOCK_RELEASE}_sb"

S3_RESSTOCK = "s3://data.sb/nrel/resstock"
S3_EIA861 = "s3://data.sb/eia/861/electric_utility_stats"

BLDG_ID = "bldg_id"
UTILITY_COL = "sb.electric_utility"
WEIGHT_COL = "weight"
ANNUAL_ELEC_COL = "out.electricity.total.energy_consumption.kwh"

PATH_METADATA = (
    f"{S3_RESSTOCK}/{RESSTOCK_RELEASE_SB}/metadata/"
    f"state={STATE_UPPER}/upgrade={UPGRADE}/metadata-sb.parquet"
)
PATH_UTILITY_ASSIGNMENT = (
    f"{S3_RESSTOCK}/{RESSTOCK_RELEASE_SB}/metadata_utility/"
    f"state={STATE_UPPER}/utility_assignment.parquet"
)
PATH_ANNUAL = (
    f"{S3_RESSTOCK}/{RESSTOCK_RELEASE_RAW}/load_curve_annual/"
    f"state={STATE_UPPER}/upgrade={UPGRADE}/"
    f"{STATE_UPPER}_upgrade{UPGRADE}_metadata_and_annual_results.parquet"
)
PATH_EIA861 = f"{S3_EIA861}/year={EIA_YEAR}/state={STATE_UPPER}/data.parquet"

print(f"State:              {STATE_UPPER}")
print(f"ResStock (raw):     {RESSTOCK_RELEASE_RAW}")
print(f"ResStock (_sb):     {RESSTOCK_RELEASE_SB}")
print(f"Upgrade:            {UPGRADE}")
print(f"EIA-861 year:       {EIA_YEAR}")
print()
print(f"Metadata:           {PATH_METADATA}")
print(f"Utility assignment: {PATH_UTILITY_ASSIGNMENT}")
print(f"Annual load curves: {PATH_ANNUAL}")
print(f"EIA-861 stats:      {PATH_EIA861}")

State:              CT
ResStock (raw):     res_2024_amy2018_2
ResStock (_sb):     res_2024_amy2018_2_sb
Upgrade:            00
EIA-861 year:       2018

Metadata:           s3://data.sb/nrel/resstock/res_2024_amy2018_2_sb/metadata/state=CT/upgrade=00/metadata-sb.parquet
Utility assignment: s3://data.sb/nrel/resstock/res_2024_amy2018_2_sb/metadata_utility/state=CT/utility_assignment.parquet
Annual load curves: s3://data.sb/nrel/resstock/res_2024_amy2018_2/load_curve_annual/state=CT/upgrade=00/CT_upgrade00_metadata_and_annual_results.parquet
EIA-861 stats:      s3://data.sb/eia/861/electric_utility_stats/year=2018/state=CT/data.parquet


## Section 1: Load data

We read three ResStock tables from S3:

1. **`metadata-sb.parquet`** — building attributes (one row per `bldg_id` for this upgrade).
2. **`utility_assignment.parquet`** — `sb.electric_utility` / `sb.gas_utility` per building.
3. **`load_curve_annual`** — one row per building with annual energy totals (including
   `out.electricity.total.energy_consumption.kwh` and sample `weight`).

After loading, we print shape, schema, and a small sample so the reader can confirm
the expected columns are present before aggregation.

In [3]:
def load_parquet(path: str) -> pl.DataFrame:
    """Collect a parquet file (or Hive directory) from S3 or local disk."""
    return cast(pl.DataFrame, pl.scan_parquet(path).collect())


def preview(name: str, df: pl.DataFrame, key_cols: list[str] | None = None) -> None:
    """Print shape, optional key-column check, schema head, and a few rows."""
    print(f"=== {name} ===")
    print(f"shape: {df.shape[0]:,} rows × {df.shape[1]} cols")
    if key_cols:
        missing = [c for c in key_cols if c not in df.columns]
        if missing:
            raise ValueError(f"{name} is missing required columns: {missing}")
        print(f"required columns present: {key_cols}")
    print("schema (first 25):")
    for col, dtype in list(df.schema.items())[:25]:
        print(f"  {col}: {dtype}")
    if len(df.schema) > 25:
        print(f"  … ({len(df.schema) - 25} more columns)")
    display(df.head(5))
    print()

In [4]:
print(f"Loading metadata from:\n  {PATH_METADATA}\n")
metadata = load_parquet(PATH_METADATA)
preview("metadata-sb", metadata, key_cols=[BLDG_ID])

Loading metadata from:
  s3://data.sb/nrel/resstock/res_2024_amy2018_2_sb/metadata/state=CT/upgrade=00/metadata-sb.parquet

=== metadata-sb ===
shape: 6,166 rows × 188 cols
required columns present: ['bldg_id']
schema (first 25):
  upgrade: Int64
  weight: Float64
  in.sqft: Int64
  in.representative_income: Float64
  in.ahs_region: String
  in.aiannh_area: String
  in.area_median_income: String
  in.ashrae_iecc_climate_zone_2004: String
  in.ashrae_iecc_climate_zone_2004_2_a_split: String
  in.bathroom_spot_vent_hour: String
  in.battery: String
  in.bedrooms: String
  in.building_america_climate_zone: String
  in.cec_climate_zone: String
  in.ceiling_fan: String
  in.census_division: String
  in.census_division_recs: String
  in.census_region: String
  in.city: String
  in.clothes_dryer: String
  in.clothes_dryer_usage_level: String
  in.clothes_washer: String
  in.clothes_washer_presence: String
  in.clothes_washer_usage_level: String
  in.cooking_range: String
  … (163 more columns

upgrade,weight,in.sqft,in.representative_income,in.ahs_region,in.aiannh_area,in.area_median_income,in.ashrae_iecc_climate_zone_2004,in.ashrae_iecc_climate_zone_2004_2_a_split,in.bathroom_spot_vent_hour,in.battery,in.bedrooms,in.building_america_climate_zone,in.cec_climate_zone,in.ceiling_fan,in.census_division,in.census_division_recs,in.census_region,in.city,in.clothes_dryer,in.clothes_dryer_usage_level,in.clothes_washer,in.clothes_washer_presence,in.clothes_washer_usage_level,in.cooking_range,in.cooking_range_usage_level,in.cooling_setpoint,in.cooling_setpoint_has_offset,in.cooling_setpoint_offset_magnitude,in.cooling_setpoint_offset_period,in.corridor,in.county,in.county_and_puma,in.county_name,in.dehumidifier,in.dishwasher,in.dishwasher_usage_level,…,in.solar_hot_water,in.state,in.tenure,in.units_represented,in.usage_level,in.utility_bill_electricity_fixed_charges,in.utility_bill_electricity_marginal_rates,in.utility_bill_fuel_oil_fixed_charges,in.utility_bill_fuel_oil_marginal_rates,in.utility_bill_natural_gas_fixed_charges,in.utility_bill_natural_gas_marginal_rates,in.utility_bill_propane_fixed_charges,in.utility_bill_propane_marginal_rates,in.utility_bill_scenario_names,in.utility_bill_simple_filepaths,in.vacancy_status,in.vintage,in.vintage_acs,in.water_heater_efficiency,in.water_heater_fuel,in.water_heater_in_unit,in.water_heater_location,in.weather_file_city,in.weather_file_latitude,in.weather_file_longitude,in.window_areas,in.windows,bldg_id,postprocess_group.has_hp,postprocess_group.heating_type,postprocess_group.heating_type_v2,heats_with_electricity,heats_with_natgas,heats_with_oil,heats_with_propane,has_natgas_connection,mf_non_hvac_electricity_adjusted
i64,f64,i64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,i64,str,i64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,i64,bool,str,str,bool,bool,bool,bool,bool,bool
0,252.301639,623,13633.0,"""Non-CBSA New England""","""No""","""0-30%""","""5A""","""5A""","""Hour3""","""None""","""1""","""Cold""","""None""","""Standard Efficiency""","""New England""","""New England""","""Northeast""","""CT, Torrington""","""Electric""","""100% Usage""","""EnergyStar""","""Yes""","""100% Usage""","""Electric Resistance""","""100% Usage""","""70F""","""No""","""0F""","""None""","""Double-Loaded Interior""","""G0900050""","""G0900050, G09000500""","""Litchfield County""","""None""","""318 Rated kWh""","""100% Usage""",…,"""None""","""CT""","""Renter""",1,"""Medium""",10,0.217925,"""0""","""3.012653846""","""11.25""","""1.397604181""","""0""","""3.365153846""","""Utility Rates - Fixed + Variab…","""data/simple_rates/State.tsv""","""Occupied""","""1950s""","""1940-59""","""Fuel Oil Standard""","""Fuel Oil""","""No""","""None""","""Waterbury Oxford""",41.48,-73.13,"""F6 B6 L6 R6""","""Double, Clear, Non-metal, Air""",24633,false,"""fossil_fuel""","""delivered_fuels""",false,false,true,false,false,true
0,252.301639,1138,63747.0,"""Non-CBSA New England""","""No""","""60-80%""","""5A""","""5A""","""Hour21""","""None""","""2""","""Cold""","""None""","""None""","""New England""","""New England""","""Northeast""","""CT, Stamford""","""Electric""","""120% Usage""","""Standard""","""Yes""","""120% Usage""","""Electric Resistance""","""120% Usage""","""72F""","""No""","""0F""","""None""","""Double-Loaded Interior""","""G0900010""","""G0900010, G09000102""","""Fairfield County""","""None""","""290 Rated kWh""","""120% Usage""",…,"""None""","""CT""","""Owner""",1,"""High""",10,0.217925,"""0""","""3.012653846""","""11.25""","""1.397604181""","""0""","""3.365153846""","""Utility Rates - Fixed + Variab…","""data/simple_rates/State.tsv""","""Occupied""","""2010s""","""2010s""","""Natural Gas Standard""","""Natural Gas""","""No""","""None""","""Bridgeport Igor I""",41.18,-73.15,"""F30 B30 L30 R30""","""Double, Low-E, Non-metal, Air,…",74282,false,"""electrical_resi

In [5]:
print(f"Loading utility assignment from:\n  {PATH_UTILITY_ASSIGNMENT}\n")
utility_assignment = load_parquet(PATH_UTILITY_ASSIGNMENT)
preview(
    "utility_assignment",
    utility_assignment,
    key_cols=[BLDG_ID, UTILITY_COL],
)

Loading utility assignment from:
  s3://data.sb/nrel/resstock/res_2024_amy2018_2_sb/metadata_utility/state=CT/utility_assignment.parquet

=== utility_assignment ===
shape: 6,166 rows × 3 cols
required columns present: ['bldg_id', 'sb.electric_utility']
schema (first 25):
  bldg_id: Int64
  sb.electric_utility: String
  sb.gas_utility: String


bldg_id,sb.electric_utility,sb.gas_utility
i64,str,str
24633,"""clp""",null
74282,"""clp""","""yankee_gas"""
417405,"""clp""",null
418808,"""clp""",null
420746,"""clp""","""yankee_gas"""


In [6]:
print(f"Loading annual load curves from:\n  {PATH_ANNUAL}\n")
annual = load_parquet(PATH_ANNUAL)
preview(
    "load_curve_annual",
    annual,
    key_cols=[BLDG_ID, WEIGHT_COL, ANNUAL_ELEC_COL],
)

Loading annual load curves from:
  s3://data.sb/nrel/resstock/res_2024_amy2018_2/load_curve_annual/state=CT/upgrade=00/CT_upgrade00_metadata_and_annual_results.parquet

=== load_curve_annual ===
shape: 6,165 rows × 112 cols
required columns present: ['bldg_id', 'weight', 'out.electricity.total.energy_consumption.kwh']
schema (first 25):
  upgrade: Int64
  weight: Float64
  out.params.door_area_ft_2: Int64
  out.params.duct_unconditioned_surface_area_ft_2: Float64
  out.params.floor_area_attic_ft_2: Float64
  out.params.floor_area_attic_insulation_increase_ft_2_delta_r_value: Float64
  out.params.floor_area_conditioned_infiltration_reduction_ft_2_delta_ach_50: Float64
  out.params.floor_area_foundation_ft_2: Float64
  out.params.floor_area_lighting_ft_2: Int64
  out.params.flow_rate_mechanical_ventilation_cfm: Int64
  out.params.rim_joist_area_above_grade_exterior_ft_2: Float64
  out.params.roof_area_ft_2: Float64
  out.params.size_cooling_system_primary_k_btu_h: Float64
  out.params.si

upgrade,weight,out.params.door_area_ft_2,out.params.duct_unconditioned_surface_area_ft_2,out.params.floor_area_attic_ft_2,out.params.floor_area_attic_insulation_increase_ft_2_delta_r_value,out.params.floor_area_conditioned_infiltration_reduction_ft_2_delta_ach_50,out.params.floor_area_foundation_ft_2,out.params.floor_area_lighting_ft_2,out.params.flow_rate_mechanical_ventilation_cfm,out.params.rim_joist_area_above_grade_exterior_ft_2,out.params.roof_area_ft_2,out.params.size_cooling_system_primary_k_btu_h,out.params.size_heat_pump_backup_primary_k_btu_h,out.params.size_heating_system_primary_k_btu_h,out.params.size_heating_system_secondary_k_btu_h,out.params.size_water_heater_gal,out.params.slab_perimeter_exposed_conditioned_ft,out.params.wall_area_above_grade_conditioned_ft_2,out.params.wall_area_above_grade_exterior_ft_2,out.params.wall_area_below_grade_ft_2,out.params.window_area_ft_2,out.electricity.ceiling_fan.energy_consumption.kwh,out.electricity.clothes_dryer.energy_consumption.kwh,out.electricity.clothes_washer.energy_consumption.kwh,out.electricity.cooling.energy_consumption.kwh,out.electricity.cooling_fans_pumps.energy_consumption.kwh,out.electricity.dishwasher.energy_consumption.kwh,out.electricity.freezer.energy_consumption.kwh,out.electricity.heating.energy_consumption.kwh,out.electricity.heating_fans_pumps.energy_consumption.kwh,out.electricity.heating_hp_bkup.energy_consumption.kwh,out.electricity.heating_hp_bkup_fa.energy_consumption.kwh,out.electricity.hot_water.energy_consumption.kwh,out.electricity.lighting_exterior.energy_consumption.kwh,out.electricity.lighting_garage.energy_consumption.kwh,out.electricity.lighting_interior.energy_consumption.kwh,…,out.hot_water.fixtures.gal,out.load.cooling.energy_delivered.kbtu,out.load.heating.energy_delivered.kbtu,out.load.hot_water.energy_delivered.kbtu,out.electricity.summer.peak.kw,out.electricity.winter.peak.kw,out.load.cooling.peak.kbtu_hr,out.load.heating.peak.kbtu_hr,out.unmet_hours.cooling.hour,out.unmet_hours.heating.hour,out.emissions.all_fuels.lrmer_high_re_cost_15.co2e_kg,out.emissions.all_fuels.lrmer_low_re_cost_15.co2e_kg,out.emissions.all_fuels.lrmer_mid_case_15.co2e_kg,out.emissions.all_fuels.lrmer_mid_case_25.co2e_kg,out.emissions.electricity.lrmer_high_re_cost_15.co2e_kg,out.emissions.electricity.lrmer_low_re_cost_15.co2e_kg,out.emissions.electricity.lrmer_mid_case_15.co2e_kg,out.emissions.electricity.lrmer_mid_case_25.co2e_kg,out.emissions.fuel_oil.lrmer_high_re_cost_15.co2e_kg,out.emissions.fuel_oil.lrmer_low_re_cost_15.co2e_kg,out.emissions.fuel_oil.lrmer_mid_case_15.co2e_kg,out.emissions.fuel_oil.lrmer_mid_case_25.co2e_kg,out.emissions.natural_gas.lrmer_high_re_cost_15.co2e_kg,out.emissions.natural_gas.lrmer_low_re_cost_15.co2e_kg,out.emissions.natural_gas.lrmer_mid_case_15.co2e_kg,out.emissions.natural_gas.lrmer_mid_case_25.co2e_kg,out.emissions.propane.lrmer_high_re_cost_15.co2e_kg,out.emissions.propane.lrmer_low_re_cost_15.co2e_kg,out.emissions.propane.lrmer_mid_case_15.co2e_kg,out.emissions.propane.lrmer_mid_case_25.co2e_kg,out.bills.all_fuels.usd,out.bills.electricity.usd,out.bills.fuel_oil.usd,out.bills.natural_gas.usd,out.bills.propane.usd,out.energy_burden.percentage,bldg_id
i64,f64,i64,f64,f64,f64,f64,f64,i64,i64,f64,f64,f64,f64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64
0,252.301639,0,0.0,0.0,0.0,0.0,0.0,322,0,0.0,0.0,6.59,0.0,8.18,0,30,0.0,299.6,299.6,0.0,54.0,67.992488,0.0,0.0,895.918262,0.0,0.0,0.0,0.0,53.632006,0.0,0.0,0.0,29.014036,0.0,244.714344,…,4735.2,8219.0,2979.0,3575.0,3.8948,3.7855,6.258,8.498,301.0,0.0,1944.396269,1832.426992,1964.576593,1909.909641,776.200871,664.231595,796.381196,741.714243,1168.195397,1168.195397,1168.195397,1168.195397,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,981.133882,696.193882,284.94,0.0,0.0,8.81,461038
0,252.30

### Align on shared buildings

Metadata, utility assignment, and annual loads should each have one row per building
for this state/upgrade. Small mismatches can occur when a building is present in
metadata / utility assignment but missing from `load_curve_annual` (or vice versa).

For the rest of this notebook we keep only the **intersection** of `bldg_id`s across
all three tables, and report any IDs that were dropped.

In [7]:
n_meta = metadata.height
n_ua = utility_assignment.height
n_annual = annual.height
n_meta_ids = metadata[BLDG_ID].n_unique()
n_ua_ids = utility_assignment[BLDG_ID].n_unique()
n_annual_ids = annual[BLDG_ID].n_unique()

print("Before intersection:")
print(f"  metadata:           {n_meta:,} rows, {n_meta_ids:,} unique {BLDG_ID}")
print(f"  utility_assignment: {n_ua:,} rows, {n_ua_ids:,} unique {BLDG_ID}")
print(f"  load_curve_annual:  {n_annual:,} rows, {n_annual_ids:,} unique {BLDG_ID}")

if n_meta != n_meta_ids:
    print("WARNING: metadata has duplicate bldg_id values")
if n_ua != n_ua_ids:
    print("WARNING: utility_assignment has duplicate bldg_id values")
if n_annual != n_annual_ids:
    print("WARNING: load_curve_annual has duplicate bldg_id values")

meta_ids = set(metadata[BLDG_ID].to_list())
ua_ids = set(utility_assignment[BLDG_ID].to_list())
annual_ids = set(annual[BLDG_ID].to_list())
shared_ids = meta_ids & ua_ids & annual_ids

dropped_meta = sorted(meta_ids - shared_ids)
dropped_ua = sorted(ua_ids - shared_ids)
dropped_annual = sorted(annual_ids - shared_ids)

print(f"\nShared bldg_ids (metadata ∩ utility_assignment ∩ annual): {len(shared_ids):,}")
print(f"  dropped from metadata only:           {len(dropped_meta):,} {dropped_meta[:10]}")
print(f"  dropped from utility_assignment only: {len(dropped_ua):,} {dropped_ua[:10]}")
print(f"  dropped from annual only:             {len(dropped_annual):,} {dropped_annual[:10]}")

shared_ids_list = list(shared_ids)
metadata = metadata.filter(pl.col(BLDG_ID).is_in(shared_ids_list))
utility_assignment = utility_assignment.filter(pl.col(BLDG_ID).is_in(shared_ids_list))
annual = annual.filter(pl.col(BLDG_ID).is_in(shared_ids_list))

print("\nAfter intersection:")
print(f"  metadata:           {metadata.height:,}")
print(f"  utility_assignment: {utility_assignment.height:,}")
print(f"  load_curve_annual:  {annual.height:,}")
assert metadata.height == utility_assignment.height == annual.height
assert metadata[BLDG_ID].n_unique() == metadata.height

metadata:           6,166 rows, 6,166 unique bldg_id
utility_assignment: 6,166 rows, 6,166 unique bldg_id
load_curve_annual:  6,165 rows, 6,165 unique bldg_id

bldg_id overlap (utility_assignment ∩ annual): 6,165


## Section 2: Sum electricity by utility

Join the aligned annual loads to utility assignment on `bldg_id`, then for each
`sb.electric_utility` sum **weighted** annual electricity:

$$\text{resstock\_total\_kwh} = \sum_i (\text{annual\_kwh}_i \times \text{weight}_i)$$

ResStock `weight` is the sample expansion factor (each building represents
roughly 252 dwellings). Customer counts are likewise $\sum_i \text{weight}_i$.

Buildings with a null electric utility assignment are reported separately and
excluded from the per-utility totals.

In [ ]:
annual_with_utility = annual.select(
    BLDG_ID,
    pl.col(ANNUAL_ELEC_COL).alias("annual_kwh"),
    pl.col(WEIGHT_COL),
).join(
    utility_assignment.select(BLDG_ID, UTILITY_COL),
    on=BLDG_ID,
    how="inner",
    validate="1:1",
).with_columns(
    (pl.col("annual_kwh") * pl.col(WEIGHT_COL)).alias("weighted_kwh"),
)

n_null_utility = annual_with_utility.filter(pl.col(UTILITY_COL).is_null()).height
if n_null_utility:
    print(f"WARNING: {n_null_utility:,} buildings have null {UTILITY_COL}; excluded from totals")

resstock_by_utility = (
    annual_with_utility.filter(pl.col(UTILITY_COL).is_not_null())
    .group_by(UTILITY_COL)
    .agg(
        pl.len().alias("buildings"),
        pl.col(WEIGHT_COL).sum().alias("resstock_customers"),
        pl.col("weighted_kwh").sum().alias("resstock_total_kwh"),
    )
    .sort("resstock_total_kwh", descending=True)
    .rename({UTILITY_COL: "utility_code"})
)

resstock_by_utility_with_total = pl.concat(
    [
        resstock_by_utility,
        pl.DataFrame(
            {
                "utility_code": ["**TOTAL**"],
                "buildings": [resstock_by_utility["buildings"].sum()],
                "resstock_customers": [resstock_by_utility["resstock_customers"].sum()],
                "resstock_total_kwh": [resstock_by_utility["resstock_total_kwh"].sum()],
            },
            schema=resstock_by_utility.schema,
        ),
    ]
)

print(f"ResStock weighted electricity by {UTILITY_COL} ({STATE_UPPER}, upgrade {UPGRADE}):")
display(resstock_by_utility_with_total)

## Section 3: Compare to EIA-861

*Coming next — load EIA-861 residential sales and compare per-utility totals.*

## Section 4: Assumptions and limitations

*Coming next — document weighting, residential-only EIA scope, year alignment, and
known discrepancy drivers.*